In [0]:
import pandas as pd

In [0]:
#Leyendo la tabla de caracteristicas generales
ft = spark.read.format("delta").table("workspace.electiva.fuerzadetrabajo")
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados")
cgse = spark.read.format("delta").table("workspace.electiva.cgse")

# Convertir el DataFrame de Spark a Pandas
ft = ft.toPandas()
desempleados = desempleados.toPandas()
cgse = cgse.toPandas()

#Primero hay que anexar la p3271 a la tabla de desempleados
desempleados = pd.merge(desempleados, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271','P6040']], 
                        on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')

ft = pd.merge(ft, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271','P6040']], 
                        on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')

In [0]:
ft.columns

In [0]:
#1. Para el total nacional calcule (a Junio de 2026):
#población en edad de trabajar
#población que no está en edad de trabajar
#población económicamente activa
#población económicamente inactiva
#tasa de ocupación
#tasa de desempleo
total  = ft['FEX_C18'].sum()
pet    = ft[ft['PET']==1]['FEX_C18'].sum()      
no_pet = total - pet                           
pea    = ft[ft['FT']==1]['FEX_C18'].sum()       
pei    = ft[ft['FFT']==1]['FEX_C18'].sum()     

desocupados = desempleados[desempleados['DSI']==1]['FEX_C18'].sum()
ocupados    = pea - desocupados

In [0]:
tasa_ocupacion  = ocupados / pet * 100
tasa_desempleo  = desocupados / pea * 100


In [0]:
print(f"Población total: {total:,.0f}")
print(f"PET: {pet:,.0f}")
print(f"No en edad de trabajar: {no_pet:,.0f}")
print(f"PEA: {pea:,.0f}")
print(f"PEI: {pei:,.0f}")
print(f"Ocupados: {ocupados:,.0f} | Desocupados: {desocupados:,.0f}")
print(f"Tasa de ocupación: {tasa_ocupacion:.2f}%")
print(f"Tasa de desempleo: {tasa_desempleo:.2f}%")
print(f"Chequeo PEA + PEI vs PET: {pea + pei:,.0f} vs {pet:,.0f}")

In [0]:
#2. Calcule las tasas de desempleo para hombres y mujeres en todos los departamentos de Colombia a Junio de 2026, ¿cuáles son los departamentos con mayores y menores tasas de desempleo?
desempleados

In [0]:
# desempleados_dpto -> cantidad de hombres y mujeres desempleados por departamento
desempleados_dpto_genero = desempleados[desempleados['DSI']==1].groupby(['P3271','DPTO'])['FEX_C18'].sum()
desempleados_dpto_genero

In [0]:
#cantidad de hombres y mujeres economicamente activos por departamento
pea_dpto_genero = ft[ft['FT']==1].groupby(['P3271','DPTO'])['FEX_C18'].sum()
pea_dpto_genero

In [0]:
tabla_desempleo = pd.DataFrame({'PEA': pea_dpto_genero, 'DESEMPLEADOS':desempleados_dpto_genero})

In [0]:
tabla_desempleo['TD']= (tabla_desempleo['DESEMPLEADOS']/tabla_desempleo['PEA'])*100

In [0]:
tabla_desempleo.head(100)

In [0]:
# 4. Diccionarios de mapeo
dpto_nombre = {
    5: 'ANTIOQUIA', 8: 'ATLANTICO', 11: 'BOGOTA', 13: 'BOLIVAR', 15: 'BOYACA',
    17: 'CALDAS', 18: 'CAQUETA', 19: 'CAUCA', 20: 'CESAR', 23: 'CORDOBA',
    25: 'CUNDINAMARCA', 27: 'CHOCO', 41: 'HUILA', 44: 'GUAJIRA', 47: 'MAGDALENA',
    50: 'META', 52: 'NARIÑO', 54: 'NORTE DE SANTANDER', 63: 'QUINDIO', 66: 'RISARALDA',
    68: 'SANTANDER', 70: 'SUCRE', 73: 'TOLIMA', 76: 'VALLE DEL CAUCA', 81: 'ARAUCA',
    85: 'CASANARE', 86: 'PUTUMAYO', 88: 'SAN ANDRES', 91: 'AMAZONAS', 94: 'GUAINIA',
    95: 'GUAVIARE', 97: 'VAUPES', 99: 'VICHADA'
}

sexo_nombre = {1:'Hombre',2:'Mujer'}

tabla_desempleo = tabla_desempleo.reset_index()

tabla_desempleo['Departamento'] = tabla_desempleo['DPTO'].map(dpto_nombre)
tabla_desempleo['Sexo'] = tabla_desempleo['P3271'].map(sexo_nombre)

In [0]:
tabla_desempleo = tabla_desempleo[['Departamento','Sexo','PEA','DESEMPLEADOS','TD']]
tabla_desempleo


In [0]:
# Ordenar por dos variables de mayor a menor (descendente)
tabla_desempleo = tabla_desempleo.sort_values(by=[ 'Sexo','TD'], ascending=False)
tabla_desempleo

In [0]:
tabla_desempleo[tabla_desempleo['Sexo']=='Hombre']

In [0]:
tabla_desempleo[tabla_desempleo['Sexo']=='Mujer']

In [0]:
for sexo in ['Hombre', 'Mujer']:
    t = tabla_desempleo[tabla_desempleo['Sexo'] == sexo].sort_values('TD', ascending=False)
    print(f"\n--- {sexo}s: 3 departamentos con MAYOR desempleo ---")
    print(t[['Departamento', 'TD']].head(3))
    print(f"--- {sexo}s: 3 departamentos con MENOR desempleo ---")
    print(t[['Departamento', 'TD']].tail(3))

In [0]:
#3. Tomando en consideración únicamente los siguientes grupos de edad:
# 18 a 24
# 24 a 30
# 30 a 40
# 40 a 50
# 60 en adelante
#Para cada uno de los departamentos, cual es el grupo de edad con mayor y menor tasa de desempleo ?
ft['GRUPO'] = pd.cut(ft['P6040'], bins=[18,24,30,40,50,60,200],
                     labels=['18-24','24-30','30-40','40-50','50-60','60+'], right=False)
desempleados['GRUPO'] = pd.cut(desempleados['P6040'], bins=[18,24,30,40,50,60,200],
                     labels=['18-24','24-30','30-40','40-50','50-60','60+'], right=False)
f = ft[ft['GRUPO'] != '50-60']      # el enunciado no incluye 50 a 60
d = desempleados[desempleados['GRUPO'] != '50-60']

pea_g = f[f['FT']==1].groupby(['DPTO','GRUPO'], observed=True)['FEX_C18'].sum()
des_g = d[d['DSI']==1].groupby(['DPTO','GRUPO'], observed=True)['FEX_C18'].sum()
des_g = des_g.reindex(pea_g.index, fill_value=0)

tasa_g = (des_g / pea_g * 100).reset_index(name='TASA')
tasa_g['DEPARTAMENTO'] = tasa_g['DPTO'].map(dpto_nombre).fillna(tasa_g['DPTO'])

mayor = tasa_g.loc[tasa_g.groupby('DPTO')['TASA'].idxmax()][['DEPARTAMENTO','GRUPO','TASA']]
menor = tasa_g.loc[tasa_g.groupby('DPTO')['TASA'].idxmin()][['DEPARTAMENTO','GRUPO','TASA']]

print("Grupo con MAYOR tasa de desempleo por departamento:"); print(mayor)
print("Grupo con MENOR tasa de desempleo por departamento:"); print(menor)

In [0]:
#4. ¿Cuáles son los 3 departamentos con menor tasa de desempleo de mujeres entre los 18 y los 25 años ?
pea_m = ft[(ft['FT']==1) & (ft['P3271']==2) & (ft['P6040']>=18) & (ft['P6040']<=25)] \
          .groupby('DPTO')['FEX_C18'].sum()

des_m = desempleados[(desempleados['DSI']==1) & (desempleados['P3271']==2) &
                     (desempleados['P6040']>=18) & (desempleados['P6040']<=25)] \
          .groupby('DPTO')['FEX_C18'].sum().reindex(pea_m.index, fill_value=0)

tasa_m = (des_m / pea_m * 100).sort_values()
tasa_m.index = [dpto_nombre.get(d, d) for d in tasa_m.index]
print(tasa_m.head(3))

In [0]:
spark.sql("SHOW TABLES IN workspace.electiva").show()

In [0]:
# 5. Calcule el promedio de ingresos laborales (variable INGLABO) de las personas residentes en los departamentos de Risaralda, Caldas , Quindío, Antioquia y Valle del Cauca.
ocupados = spark.read.format("delta").table("workspace.electiva.ocupados").toPandas()

# Si no trae sexo o edad, los traemos de cgse (igual que hiciste con las otras tablas)
faltan = [c for c in ['P3271','P6040'] if c not in ocupados.columns]
if faltan:
    ocupados = pd.merge(ocupados, cgse[['DIRECTORIO','SECUENCIA_P','ORDEN'] + faltan],
                        on=['DIRECTORIO','SECUENCIA_P','ORDEN'], how='left')

# Asegurar que sean números
for c in ['DPTO','INGLABO','FEX_C18','P6040','P3271']:
    ocupados[c] = pd.to_numeric(ocupados[c], errors='coerce')

print(ocupados.columns.tolist())

In [0]:
dptos = {66:'Risaralda', 17:'Caldas', 63:'Quindío', 5:'Antioquia', 76:'Valle del Cauca'}

oc5 = ocupados[(ocupados['DPTO'].isin(dptos.keys())) & (ocupados['INGLABO']>0)].copy()

# Promedio ponderado con el factor de expansión
oc5['ING_POND'] = oc5['INGLABO'] * oc5['FEX_C18']
prom = oc5.groupby('DPTO')['ING_POND'].sum() / oc5.groupby('DPTO')['FEX_C18'].sum()
prom.index = prom.index.map(dptos)

print(prom.sort_values(ascending=False).round(0))

In [0]:
#6. Para los mismos departamentos del punto anterior, calcule el promedio de ingresos laborales de hombres y mujeres en los siguientes rangos de edad :
#18 a 24
#24 a 30
#30 a 40
#40 a 50
#60 en adelante
bins = [18, 24, 30, 40, 50, 60, 200]
labels = ['18-24','24-30','30-40','40-50','50-60','60+']
oc5['GRUPO'] = pd.cut(oc5['P6040'], bins=bins, labels=labels, right=False)
oc6 = oc5[oc5['GRUPO'] != '50-60']

num = oc6.groupby(['DPTO','P3271','GRUPO'], observed=True)['ING_POND'].sum()
den = oc6.groupby(['DPTO','P3271','GRUPO'], observed=True)['FEX_C18'].sum()
prom6 = (num / den).unstack('GRUPO').round(0)

prom6.index = [(dptos[d], {1:'Hombre',2:'Mujer'}[s]) for d, s in prom6.index]
print(prom6)

In [0]:
print([c for c in ocupados.columns if 'OFICIO' in c])

In [0]:
#7.¿Cuáles son los oficios con mayores promedios salariales a nivel nacional en general ? 
OFICIO = 'OFICIO_C8'
ocupados[OFICIO] = pd.to_numeric(ocupados[OFICIO], errors='coerce')

oc = ocupados[ocupados['INGLABO'] > 0].copy()
oc['ING_POND'] = oc['INGLABO'] * oc['FEX_C18']

g = oc.groupby(OFICIO).agg(ING=('ING_POND','sum'), PESO=('FEX_C18','sum'), N=('INGLABO','size'))
g['PROMEDIO'] = g['ING'] / g['PESO']

top = g[g['N'] >= 30].sort_values('PROMEDIO', ascending=False).head(10)
print(top[['PROMEDIO','N']].round(0))

In [0]:
#8. Elija un código de oficio y , basado en los datos, concluya si existen diferencias significativas entre los promedios salariales de hombres y mujeres para dicho oficio en los departamentos de Caldas , Quindío, Antioquia y Valle del Cauca.
from scipy import stats

OFICIO_ELEGIDO = 5223
dptos4 = {17:'Caldas', 63:'Quindío', 5:'Antioquia', 76:'Valle del Cauca'}

for d, nombre in dptos4.items():
    s = oc[(oc[OFICIO] == OFICIO_ELEGIDO) & (oc['DPTO'] == d)]
    h = s[s['P3271'] == 1]['INGLABO']      # hombres
    m = s[s['P3271'] == 2]['INGLABO']      # mujeres
    if len(h) > 1 and len(m) > 1:
        t, p = stats.ttest_ind(h, m, equal_var=False)
        print(f"{nombre}: hombres ${h.mean():,.0f} (n={len(h)}) | mujeres ${m.mean():,.0f} (n={len(m)}) | p-valor={p:.3f} -> "
              f"{'diferencia SIGNIFICATIVA' if p < 0.05 else 'sin diferencia significativa'}")
    else:
        print(f"{nombre}: muy pocos datos (hombres={len(h)}, mujeres={len(m)})")

In [0]:
#9. En el departamento del Quindío cual es el oficio con mayor y menor promedio salarial en hombres y mujeres?
q = oc[oc['DPTO'] == 63]
t9 = q.groupby(['P3271', OFICIO]).agg(ING=('ING_POND','sum'), PESO=('FEX_C18','sum'), N=('INGLABO','size'))
t9['PROMEDIO'] = t9['ING'] / t9['PESO']
t9 = t9[t9['N'] >= 5].reset_index()      # mínimo 5 personas porque el Quindío es pequeño

for sexo, nombre in [(1,'Hombres'), (2,'Mujeres')]:
    s = t9[t9['P3271'] == sexo].sort_values('PROMEDIO')
    print(f"\n{nombre} en el Quindío")
    print("MAYOR promedio:", int(s.iloc[-1][OFICIO]), f"${s.iloc[-1]['PROMEDIO']:,.0f}", f"(n={int(s.iloc[-1]['N'])})")
    print("MENOR promedio:", int(s.iloc[0][OFICIO]),  f"${s.iloc[0]['PROMEDIO']:,.0f}",  f"(n={int(s.iloc[0]['N'])})")